<a href="https://colab.research.google.com/github/Nakib-Nasrullah/Dengue_Research/blob/main/Hybrid_Ensemble.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Imoort libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import warnings
warnings.filterwarnings('ignore')

Mounted at /content/drive


In [3]:
df = pd.read_csv('/content/drive/MyDrive/Datasets/Dengue_final_dataset_updated.csv')

In [4]:
# Clean columns
df.columns = [c.strip() for c in df.columns]

In [10]:
# Detect target column
target_col = None
for col in ['Dengue_case']:
    if col in df.columns:
        target_col = col
        break

In [11]:
# Date handling
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values('Date')

In [12]:
# Fill missing
df = df.ffill().bfill()

# **Feature Engineering (Lag Features)**

In [13]:
def create_lag(df, col, lags):
    for lag in lags:
        df[f"{col}_Lag_{lag}"] = df[col].shift(lag)
    return df

df = create_lag(df, target_col, [14,21])

if 'Temperature' in df.columns:
    df = create_lag(df, 'Temperature', [14,21,30])

df = df.bfill()


Feature Selection (XGBoost Importance)

In [14]:
from xgboost import XGBRegressor

features = [c for c in df.columns if 'Lag' in c or c in ['Rainfall','Humidity','Temperature']]
X = df[features]
y = df[target_col]

model_fs = XGBRegressor()
model_fs.fit(X, y)

import pandas as pd

importance = pd.DataFrame({
    'Feature': features,
    'Importance': model_fs.feature_importances_
}).sort_values(by='Importance', ascending=False)

print(importance)

# Select top features
selected_features = importance['Feature'].head(6).tolist()
print("Selected:", selected_features)


               Feature  Importance
14        Dengue_Lag_7    0.096950
3      Rainfall_Lag_14    0.091356
5      Rainfall_Lag_30    0.072691
9          Temp_Lag_30    0.068904
4      Rainfall_Lag_21    0.065675
8          Temp_Lag_21    0.062382
1             Rainfall    0.059371
10      Humidity_Lag_7    0.058102
11     Humidity_Lag_14    0.054019
6           Temp_Lag_7    0.053723
13     Humidity_Lag_30    0.051060
16  Dengue_case_Lag_21    0.048677
7          Temp_Lag_14    0.047072
0             Humidity    0.046823
2       Rainfall_Lag_7    0.046743
15  Dengue_case_Lag_14    0.046274
12     Humidity_Lag_21    0.030178
Selected: ['Dengue_Lag_7', 'Rainfall_Lag_14', 'Rainfall_Lag_30', 'Temp_Lag_30', 'Rainfall_Lag_21', 'Temp_Lag_21']
